> https://docs.langchain.com/oss/python/langgraph/add-memory

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# 1. 모델 초기화
model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    temperature=0
)

In [5]:
# uv add langgraph-checkpoint-sqlite
# https://sqlitebrowser.org/dl/
from typing import TypedDict, Annotated
import operator

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState
# from langgraph.checkpoint.memory import InMemorySaver
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver # InMemorySaver 대신 sqlite3과 SqliteSaver를 import 합니다.

# 2. State 정의
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# 3. Node 정의
def llm_node(state: MessagesState):
    response = model.invoke(
        state["messages"]
    )
    return {"messages": [response]}

# 4. Graph 생성
graph_builder = StateGraph(MessagesState)

# 5. Graph에 Node 추가
graph_builder.add_node("llm", llm_node)

# 6. Edge 추가하여 Node 연결
graph_builder.add_edge(START, "llm")
graph_builder.add_edge("llm", END)

### SQLite 파일 3형제에 대한 핵심 요약

- checkpoints.db (본체): 실제 데이터가 최종적으로 저장되는 메인 보관함입니다.
- checkpoints.db-wal (작업장): 성능 향상을 위해 변경 사항을 메인 파일에 옮기기 전 임시로 적어두는 노트입니다.
- checkpoints.db-shm (지도): 여러 프로그램이 동시에 접속할 때 작업장(wal)의 어디에 데이터가 있는지 알려주는 인덱스 지도입니다.

In [6]:
# 7. Graph를 실행 가능한 형태로 컴파일
# checkpointer = InMemorySaver()
checkpointer = SqliteSaver(sqlite3.connect("./db/checkpoints.db", check_same_thread=False))
graph = graph_builder.compile(checkpointer=checkpointer)

# 8. Graph 실행
config = {"configurable": {"thread_id": "conversation_1"}}

In [ ]:
# 내 이름 알려주기
human_message = HumanMessage(content="안녕! 난 김일남이야.")
initial_message = {"messages": [human_message]}
graph.invoke(initial_message, config=config)

# 내 이름 물어보기
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


================================ Human Message =================================

안녕! 난 김일남이야.
================================== Ai Message ==================================

안녕하세요, 김일남님! 만나서 반갑습니다. 무엇을 도와드릴까요?
================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

김일남님입니다.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [8]:
config = {"configurable": {"thread_id": "conversation_2"}}

In [9]:
# 내 이름 물어보기
human_message = HumanMessage(content="내 이름이 뭐야?")
initial_message = {"messages": [human_message]}
result = graph.invoke(initial_message, config=config)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

저는 대규모 언어 모델이며, 이름이 없습니다.
